In [1]:
from transformers import pipeline

classificador = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")

frases = [
    "Eu gosto muito de One Piece",
    "Estou aprendendo mais sobre LLM e está sendo interessante",
    "Achei o final de Boku no Hero meio chato"

]

resultados = classificador(frases)

for frase, resultado in zip(frases, resultados):
    print(f"'{frase}' → {resultado['label']} (confiança: {resultado['score']:.2%})")
    

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

'Eu gosto muito de One Piece' → 5 stars (confiança: 52.32%)
'Estou aprendendo mais sobre LLM e está sendo interessante' → 3 stars (confiança: 39.33%)
'Achei o final de Boku no Hero meio chato' → 5 stars (confiança: 31.25%)


In [11]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

modelo_nome = "distilbert-base-cased-distilled-squad"
tokenizer = AutoTokenizer.from_pretrained(modelo_nome)
modelo = AutoModelForQuestionAnswering.from_pretrained(modelo_nome)

context = "It is true that all children are special, simply because they are children. But most adults are not special, and children end up as adults pretty quickly. Life then can be difficult and even disappointing. The shock of this may account for the emergence of the “snowflake generation” of university students, who are so delicate they can’t handle controversial ideas being put forward in their lectures. The roots of this fragility run deep in modern culture. So, an approach of the world that states: ''Life is wonderful, you are special and, if you are a good boy/girl, life will be amazing forever'' is not a message designed to aid bouncing back from failure or confronting catastrophe. Resilience is not about feeding ego — telling your children how wonderful they are — but strengthening it."
question = "The expression 'snowflake generation' is used to:"

inputs = tokenizer(question, context, return_tensors="pt")

with torch.no_grad():
    outputs = modelo(**inputs)

inicio = torch.argmax(outputs.start_logits)
fim = torch.argmax(outputs.end_logits) + 1
resposta = tokenizer.decode(inputs["input_ids"][0][inicio:fim])
print(resposta)



Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [37]:
for k in inputs.keys():
    print(inputs[k])


tensor([[  101,  1109,  2838,   112,  4883,  2087, 13238,  3964,   112,  1110,
          1215,  1106,   131,   102,  1135,  1110,  2276,  1115,  1155,  1482,
          1132,  1957,   117,  2566,  1272,  1152,  1132,  1482,   119,  1252,
          1211,  6323,  1132,  1136,  1957,   117,  1105,  1482,  1322,  1146,
          1112,  6323,  2785,  1976,   119,  2583,  1173,  1169,  1129,  2846,
          1105,  1256, 16703,   119,  1109,  4900,  1104,  1142,  1336,  3300,
          1111,  1103, 15351,  1104,  1103,   789,  4883,  2087, 13238,  3964,
           790,  1104,  2755,  1651,   117,  1150,  1132,  1177, 10141,  1152,
          1169,   787,   189,  4282,  6241,  4133,  1217,  1508,  1977,  1107,
          1147,  9548,   119,  1109,  6176,  1104,  1142,   175, 20484, 13378,
          1576,  1996,  1107,  2030,  2754,   119,  1573,   117,  1126,  3136,
          1104,  1103,  1362,  1115,  2231,   131,   112,   112,  2583,  1110,
          7310,   117,  1128,  1132,  1957,  1105,  

In [12]:
import torch

start_logits = outputs.start_logits[0]
end_logits = outputs.end_logits[0]

melhor_score = -float("inf")
melhor_inicio, melhor_fim = 0, 0

for i in range(len(start_logits)):
    for j in range(i, min(i + 30, len(end_logits))):  # limita resposta a até 30 tokens
        score = start_logits[i] + end_logits[j]
        if score > melhor_score:
            melhor_score = score
            melhor_inicio, melhor_fim = i, j

resposta = tokenizer.decode(inputs["input_ids"][0][melhor_inicio:melhor_fim + 1])
print(resposta)

Life is wonderful, you are special and, if you are a good boy / girl, life will be amazing forever
